# EXP-LATEX-01 — LaTeX-aware ingestion vs PDF-only (Kaggle, T4 x2)

**Scientific question.** Does LaTeX-aware structural ingestion improve scientific
evidence extraction over the existing PDF-only pipeline, without degrading
correctness, provenance, attribution, abstention, reproducibility or runtime?

**The valid comparison is within this notebook**: PDF-only vs LaTeX-aware on the
*same* Kaggle hardware. Do not difference these numbers against the project's
historical RTX 3050 results — that comparison is confounded by hardware.

**Before you trust any number here**, read
`experiments/document_evidence_pipeline/LATEX_ACQUISITION_REPORT.md`. The LaTeX
ingestion arm has already been measured twice on the canonical corpus and was
**negative** both times; this notebook re-measures it under controlled
conditions and adds the previously untested atomic-table arm.

Settings -> Accelerator -> **GPU T4 x2**, and Internet **on** (BGE-M3 download).


In [ ]:
!nvidia-smi
!pip -q install sentence-transformers chromadb pymupdf 2>&1 | tail -2

## 0. Clone the repository at the experiment commit

In [ ]:
import os
REPO = "/kaggle/working/researchgpt-pipeline"
BRANCH = "claude/busy-rubin-q2j212"   # the EXP-LATEX-01 branch
if not os.path.exists(REPO):
    !git clone --branch $BRANCH https://github.com/Prakash-Ravi11/researchgpt-pipeline.git $REPO
%cd $REPO
!git rev-parse HEAD

## 1. Part 7 — environment capture

Records GPU name, count, VRAM, CUDA, driver, Python, PyTorch, embedding library
and parser versions. Everything downstream is interpreted against this record.

In [ ]:
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py --stage env

## 2. Corpus

The canonical 60-paper corpus is **not** in the repository (`data/` is
gitignored). Attach it as a Kaggle Dataset and point `--corpus` at the
`collected_papers.json` manifest. Without it, no arm can run — the runner will
refuse rather than produce partial output.

In [ ]:
CORPUS = "/kaggle/input/researchgpt-corpus/collected_papers.json"
import os, json
assert os.path.exists(CORPUS), (
    "Attach the canonical corpus dataset before running the arms. "
    "EXP-LATEX-01 does not synthesise a corpus.")
papers = json.load(open(CORPUS))
papers = papers.get("papers", papers) if isinstance(papers, dict) else papers
print(len(papers), "papers")

## 3. Part 8 — 10-paper benchmark

Per-stage timings, chunk/table/figure/equation counts, peak VRAM and peak RAM,
for both arms. Writes `benchmark_metrics.json` and `benchmark_metrics.csv`.

In [ ]:
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py \
    --stage benchmark --corpus $CORPUS --with-embedding --batch-size 8

In [ ]:
import pandas as pd
df = pd.read_csv("runs/exp-latex-01/metrics/benchmark_metrics.csv")
df.groupby("arm")[["ingest_total_s","n_chunks","n_tables","tables_split",
                   "tables_oversized","peak_ram_mib"]].agg(["mean","max"])

## 4. Part 9 — full 60-paper CONTROL (`RQ_LATEX_CHUNKING=0`, `RQ_TABLE_ATOMIC=0`)

This is measured here, on this hardware. The project's historical 34/60 and 8.9 %
figures are **not** used as the control.

In [ ]:
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py \
    --stage arm --arm control --corpus $CORPUS

## 5. Part 10 — full 60-paper TREATMENT (`RQ_LATEX_CHUNKING=1`, `RQ_TABLE_ATOMIC=1`)

Control results are never deleted or overwritten — separate output directory.

In [ ]:
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py \
    --stage arm --arm latex --corpus $CORPUS

## 6. Factorial arms (recommended)

The two flags are separable, and the protocol's 2-arm design cannot attribute an
effect to either one. These two extra arms decompose it:

* `atomic` — atomic tables alone (the genuinely untested hypothesis)
* `latex_only` — LaTeX ingestion alone (reproduces the configuration that
  already measured negative in Phase 4/5x)

In [ ]:
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py --stage arm --arm atomic --corpus $CORPUS
!python experiments/EXP-LATEX-01/kaggle/kaggle_exp_latex_01.py --stage arm --arm latex_only --corpus $CORPUS

## 7. Parts 12-14 — paired analysis

Produces `metrics_summary.csv`, `paper_level_results.csv`, the per-metric
statistics JSON and the plots. Metrics with no paired observations are written
as `NOT MEASURED` — never imputed.

In [ ]:
!python experiments/EXP-LATEX-01/analyze.py --control control --treatment latex

In [ ]:
import pandas as pd
s = pd.read_csv("runs/exp-latex-01/metrics_summary.csv")
s[["metric_label","n_paired","control","treatment","absolute_delta",
   "p_value","p_adjusted","effect_size","gate","interpretation"]]

## 8. Gate check (Part 15)

Gates are decision criteria set in advance, not predictions. Report failures
honestly; do not adjust a metric to pass one.

In [ ]:
!python experiments/EXP-LATEX-01/gates.py --summary runs/exp-latex-01/metrics_summary.csv

## 9. Save artefacts

Everything worth keeping lives under `runs/exp-latex-01/`.

In [ ]:
!tar -czf /kaggle/working/exp-latex-01-artifacts.tar.gz runs/exp-latex-01
!ls -lh /kaggle/working/exp-latex-01-artifacts.tar.gz